# Setup

In [34]:
import os
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np


pio.templates.default = "plotly_dark" 



In [35]:
def load_experiment_data(experiment_name):
    """Loads the results.csv from the specified experiment folder."""
    file_path = f"../results/{experiment_name}/results.csv"
    
    if not os.path.exists(file_path):
        print(f"Could not find data for '{experiment_name}'")
        return None
        
    return pd.read_csv(file_path)



In [ ]:
# Load the data into a dictionary for easy looping later
experiment_folders = {
    "Pure Greedy": "greedy_bandit",
    "Epsilon-Greedy (10%)": "egreedysimple_bandit",
    "Epsilon-Decay": "egreedyadvanced_bandit",
    "Policy Gradient": "gradient_bandit"
}

experiments = {
    name: load_experiment_data(folder) for name, folder in experiment_folders.items()
}

# Drop any that failed to load
experiments = {k: v for k, v in experiments.items() if v is not None}

In [41]:
raw_data = {}
for name, folder in experiment_folders.items():
    raw_path = os.path.join("..", "results", folder, "raw.npz")
    if os.path.exists(raw_path):
        raw_data[name] = np.load(raw_path)
    else:
        print(f"Warning: no raw.npz for '{name}' at {raw_path}")

# Analysis

In [51]:
fig_reward = go.Figure()

for name, df in experiments.items():
    fig_reward.add_trace(go.Scatter(
        y=df['Average_Reward'],
        mode='lines',
        name=name,
        line=dict(width=2)
    ))

fig_reward.update_layout(
    title="Average Reward over Time (Multi-Armed Bandit)",
    xaxis_title="Steps",
    yaxis_title="Average Reward",
    legend_title="Agent Strategy",
    hovermode="x unified" # This is the magic Plotly setting that shows all values on hover!
)

fig_reward.show()

In [ ]:
fig_optimal = go.Figure()
 
for name, df in experiments.items():
    if 'Optimal_Action_Percentage' in df.columns:
        fig_optimal.add_trace(go.Scatter(
            y=df['Optimal_Action_Percentage'],
            mode='lines',
            name=name,
            line=dict(width=2)
        ))

fig_optimal.update_layout(
    title="Optimal Action Identification over Time",
    xaxis_title="Steps",
    yaxis_title="% Optimal Action Chosen",
    yaxis=dict(range=[0, 100]), # Lock Y-axis to 0-100%
    legend_title="Agent Strategy",
    hovermode="x unified"
)

fig_optimal.show()

In [57]:
fig_regret = go.Figure()

for name in experiments.keys():
    if name not in raw_data or 'optimal_values' not in raw_data[name]:
        continue

    reward_hist = raw_data[name]['reward_history']       # (seeds, steps)
    optimal_vals = raw_data[name]['optimal_values']       # (seeds,)

    # per-step regret per seed, then cumulative sum along steps
    per_step_regret = optimal_vals[:, None] - reward_hist  # broadcast (seeds, steps)
    cum_regret = np.cumsum(per_step_regret, axis=1)        # (seeds, steps)

    mean_regret = cum_regret.mean(axis=0)
    seeds_n = cum_regret.shape[0]
    sem_regret = cum_regret.std(axis=0, ddof=1) / np.sqrt(seeds_n)

    fig_regret.add_trace(go.Scatter(
        y=mean_regret, mode='lines', name=name, line=dict(width=2)
    ))
    line_color = fig_regret.data[-1].line.color
    fig_regret.add_trace(go.Scatter(
        y=mean_regret + 1.96 * sem_regret, mode='lines',
        line=dict(width=0), showlegend=False, hoverinfo='skip'
    ))
    fig_regret.add_trace(go.Scatter(
        y=mean_regret - 1.96 * sem_regret, mode='lines',
        line=dict(width=0), fill='tonexty',
        fillcolor='rgba(128,128,128,0.15)',
        showlegend=False, hoverinfo='skip'
    ))

fig_regret.update_layout(
    title="Cumulative Regret over Time (lower is better)",
    xaxis_title="Steps",
    yaxis_title="Cumulative Regret",
    legend_title="Agent Strategy",
    hovermode="x unified"
)

fig_regret.show()